# KV cache: prefill and decode

How much work does caching save, and how much memory does the cache need?
Run commands are in [README.md](README.md). This notebook only reads results.
Compare runs with the same model, device and token counts. No results are included yet.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from llm_systems_lab.inference_results import load_inference_results

root = Path.cwd()
if root.name == "04_kv_cache":
    root = root.parent.parent
results_dir = root / "experiments/04_kv_cache/results"


In [ ]:
runs, summary = load_inference_results(sorted(results_dir.rglob("*.json")))
if not summary.empty:
    display(summary.round(3))
else:
    print("No results yet. Run the no-cache baseline from README.md first.")


In [ ]:
selected = "baseline"  # Use p32 or p128 after collecting the other configs.
names = [name for name in runs if Path(name).name.startswith(selected + "_")]

if names:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    summary.loc[names, ["prefill_mean_ms", "prefill_median_ms"]].plot.bar(ax=axes[0])
    axes[0].set(title="Prefill", ylabel="Time [ms]", xlabel="")
    for name in names:
        result = runs[name]
        samples = pd.DataFrame(result["decode"]["samples_seconds"]) * 1000
        lengths = result["decode"]["context_lengths"]
        line, = axes[1].plot(lengths, samples.mean(), label=name)
        axes[1].plot(lengths, samples.median(), "--", color=line.get_color())
    axes[1].set(title="Decode: mean (solid), median (dashed)",
                xlabel="Context length [tokens]", ylabel="Time per batch step [ms]")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


Prefill processes the prompt. Each decode call consumes one fixed continuation token.
Without cache, the model recomputes the full prefix. Tokens/s counts the whole batch.

Cache MiB is the logical size of the final K/V tensors, excluding weights and temporaries.
It is not peak memory. A zero in no-cache mode does not mean zero memory use.
